# Verifying and re-running the Sawin parameter certificate

This notebook demonstrates the notebook-facing API for `unit-distance-gs-optimiser`. It works from the repository root or from the `notebooks/` directory, and it also bootstraps `src/` so a fresh clone can run before installation.

## Setup

For normal use, install the package with

```bash
pip install -e ".[notebook]"
```

The cell below still adds `src/` to `sys.path`, which makes the notebook robust inside VS Code and in a fresh checkout.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from IPython.display import Markdown

from unit_distance_gs_optimiser.notebook import (
    exponent_markdown,
    load_certificate,
    load_certificate_summary,
    markdown_summary_table,
    search_t_summary,
)

BASELINE_CERTIFICATE = ROOT / "certificates" / "t81-first-odd-primes.json"
IMPROVED_CERTIFICATE = ROOT / "certificates" / "t81-swap-197-337-to-433-601.json"

## Verify the improved stored certificate

In [2]:
summary = load_certificate_summary(IMPROVED_CERTIFICATE)
summary

{'valid': True, 'errors': [], 'capacity': 1518, 'total_weight': 1518, 'selected_count': 1278, 'split_count': 240, 'max_selected_prime': 16007, 'numerator': 170.95135324967507, 'denominator': 5384.152468939053, 'delta': 0.03175083808192398, 'exponent': 1.031750838081924, 'certificate_name': 't81-swap-197-337-to-433-601', 'T_size': 81, 'last_prime_in_T': 601, 'R': 32.49523163513887}

In [3]:
Markdown(markdown_summary_table(summary))

| key | value |
|---|---:|
| `valid` | `True` |
| `errors` | `[]` |
| `capacity` | `1518` |
| `total_weight` | `1518` |
| `selected_count` | `1278` |
| `split_count` | `240` |
| `max_selected_prime` | `16007` |
| `numerator` | `170.95135324967507` |
| `denominator` | `5384.152468939053` |
| `delta` | `0.03175083808192398` |
| `exponent` | `1.031750838081924` |
| `certificate_name` | `t81-swap-197-337-to-433-601` |
| `T_size` | `81` |
| `last_prime_in_T` | `601` |
| `R` | `32.49523163513887` |

## Compare with the prefix-81 certificate

The new certificate uses the non-prefix set $T=(\text{first }81\text{ odd primes})\setminus\{197,337\}\cup\{433,601\}$.

In [4]:
baseline_summary = load_certificate_summary(BASELINE_CERTIFICATE)
{
    "prefix_exponent": baseline_summary["exponent"],
    "improved_exponent": summary["exponent"],
    "delta_gain": summary["delta"] - baseline_summary["delta"],
    "split_count_prefix": baseline_summary["split_count"],
    "split_count_improved": summary["split_count"],
}

{'prefix_exponent': 1.0317221200362845, 'improved_exponent': 1.031750838081924, 'delta_gain': 2.8718045639468393e-05, 'split_count_prefix': 254, 'split_count_improved': 240}

## Re-run the optimiser for the improved `T`

This recomputes `S_Q`, the exponents `k(p)`, and `R` for the same non-prefix `T`. It is slower than certificate verification but should finish in a few seconds on a typical laptop.

In [5]:
certificate = load_certificate(IMPROVED_CERTIFICATE)
found_summary = search_t_summary(certificate.t, prime_limit=200_000)
found_summary

{'valid': True, 'errors': [], 'capacity': 1518, 'total_weight': 1518, 'selected_count': 1278, 'split_count': 240, 'max_selected_prime': 16007, 'numerator': 170.95135324967507, 'denominator': 5384.152468939053, 'delta': 0.03175083808192398, 'exponent': 1.031750838081924, 'certificate_name': 'explicit-T-81-exponent-1.031750838082', 'T_size': 81, 'last_prime_in_T': 601, 'R': 32.49523163513887, 'search_prime_limit': 200000, 'positivity_threshold': 55017.990156718566, 'candidate_limit_exceeds_threshold': True}

In [6]:
Markdown(exponent_markdown(summary, found_summary))

### Headline exponent

Stored certificate: `delta = 0.031750838081924`, 
so `1 + delta = 1.031750838081924`.

Re-run optimiser: `delta = 0.031750838081924`, 
so `1 + delta = 1.031750838081924`.

## Optional: heuristic swap search

The CLI includes a heavier heuristic search over non-prefix choices of `T`:

```bash
unit-distance-gs swap-search \
  --t-size 81 \
  --prime-limit 200000 \
  --add-prime-limit 1300 \
  --steps 2 \
  --top-swaps 2 \
  --name t81-swap-search \
  --write-certificate certificates/t81-swap-search.json
```

The stored improved certificate is the lightweight way to verify the final exponent.